# Практическое занятие № 10 — оценка классификатора и цена ошибок
**Дисциплина:** Технологии искусственного интеллекта — 1 (ТюмГУ)  
**Время:** 90 минут | **Среда:** Python, scikit-learn, Google Colab  
**Кейс:** бинарная классификация поздней доставки.  
**Результат:** матрица ошибок, метрики, стоимость C(t)=200·FP+2000·FN, выбор порога по validation и **однократная** итоговая проверка на test.  

**Оговорка:** 200 ₽ и 2000 ₽ — условные учебные потери, их нельзя выдавать за установленные экономические эффекты бизнеса.

## Шаг 1. Схема данных и воспроизводимая модель (0–15 мин)
Данные разделяем точно по той же процедуре, что и на занятии 9, либо используем `split_ids_06.csv`. Подбор модели и порога по test запрещён.

In [ ]:
# Общая подготовка: одинаковая для практик 7 и 8
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PATH = Path('orders_dirty.csv')
if not PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv в файлы Colab или измените PATH.')

df_raw = pd.read_csv(PATH, encoding='utf-8-sig')
expected = {'order_id','created_at','distance_km','zone','queue_size',
            'weather','courier_load','promised_minutes','actual_minutes','late'}
missing_cols = expected - set(df_raw.columns)
assert not missing_cols, f'Отсутствуют необходимые поля: {sorted(missing_cols)}'
print('Сырой набор:', df_raw.shape, '; полные дубликаты:', int(df_raw.duplicated().sum()))

# Удаление полных повторов: не обучаем статистику и не используем test.
df = df_raw.drop_duplicates().copy()
df['late'] = pd.to_numeric(df['late'], errors='coerce')
actual = pd.to_numeric(df['actual_minutes'], errors='coerce')
promised = pd.to_numeric(df['promised_minutes'], errors='coerce')
valid_rule = actual.notna() & promised.notna() & df['late'].isin([0, 1])
conflict = valid_rule & (df['late'] != ((actual - promised) > 15).astype(int))
print('Конфликтных меток (проверяемых):', int(conflict.sum()))
df = df.loc[~conflict & df['late'].isin([0, 1])].copy()
df['late'] = df['late'].astype(int)
# Неполные / конфликтные сведения об идентификаторе требуют отдельного решения.
assert df['order_id'].notna().all(), 'Обнаружены пустые ID заказов.'
assert not df['order_id'].duplicated().any(), 'Есть повторные ID: требуется анализ причины, не случайное удаление.'
print('Рабочая копия:', df.shape)

split_path = Path('split_ids_06.csv')  # Необязательный экспорт из практики № 6
if split_path.exists():
    splits = pd.read_csv(split_path, dtype={'order_id': str})
    assert {'order_id', 'split'} <= set(splits.columns)
    assert set(splits['split']) <= {'train', 'val', 'test'}
    assert splits['order_id'].is_unique
    x = df.assign(order_id=df['order_id'].astype(str)).merge(
        splits[['order_id', 'split']], on='order_id', how='left', validate='one_to_one')
    assert x['split'].notna().all(), 'В split_ids_06.csv перечислены не все рабочие заказы.'
    assert set(splits['order_id']) == set(x['order_id'])
    train = x.loc[x['split'] == 'train'].drop(columns='split').copy()
    val   = x.loc[x['split'] == 'val'].drop(columns='split').copy()
    test  = x.loc[x['split'] == 'test'].drop(columns='split').copy()
else:
    # Воссоздание схемы практики № 6, если файл с IDs не был сохранён.
    train_dev, test = train_test_split(df, test_size=.20, stratify=df['late'], random_state=42)
    train, val = train_test_split(train_dev, test_size=.25,
                                 stratify=train_dev['late'], random_state=42)
    split_export = pd.concat([
       train[['order_id']].assign(split='train'),
       val[['order_id']].assign(split='val'),
       test[['order_id']].assign(split='test')], ignore_index=True)
    split_export.to_csv('split_ids_06.csv', index=False)

sets = [set(a['order_id']) for a in (train, val, test)]
assert len(sets[0]&sets[1]) == len(sets[0]&sets[2]) == len(sets[1]&sets[2]) == 0
assert sets[0]|sets[1]|sets[2] == set(df['order_id'])
assert len(train)+len(val)+len(test) == len(df)
print('train / val / test:', len(train), len(val), len(test))

In [ ]:
# В Colab функция экспортируется в обычный модуль для безопасного joblib.dump.
# При перезапуске блокнота модуль восстанавливается автоматически.
from pathlib import Path
Path('tyumgu_feature_rules.py').write_text("# Правила преобразования признаков для занятий 7–10\nimport numpy as np\nimport pandas as pd\nNUM_BASE = ['distance_km', 'queue_size', 'courier_load', 'promised_minutes']\nCAT = ['zone', 'weather']\nNUM = NUM_BASE + ['hour_sin', 'hour_cos', 'planned_speed_kmh']\nINPUT = NUM_BASE + CAT + ['created_at']\n\ndef make_features(X):\n    f = X.loc[:, INPUT].copy()\n    for name in NUM_BASE:\n        f[name] = pd.to_numeric(f[name], errors='coerce')\n    for name in NUM_BASE:\n        f.loc[f[name] < 0, name] = np.nan\n    f.loc[f['promised_minutes'] == 0, 'promised_minutes'] = np.nan\n    for name in CAT:\n        s = f[name].astype('string').str.strip()\n        s = s.mask(s.str.lower().isin(['', 'nan', 'none', 'нет данных', 'неизвестно', 'null']))\n        f[name] = s.astype(object).where(s.notna(), np.nan)\n    time = pd.to_datetime(f['created_at'], errors='coerce')\n    hour = time.dt.hour.astype(float)\n    f['hour_sin'] = np.sin(2*np.pi*hour/24)\n    f['hour_cos'] = np.cos(2*np.pi*hour/24)\n    f['planned_speed_kmh'] = 60*f['distance_km']/f['promised_minutes']\n    f['planned_speed_kmh'] = f['planned_speed_kmh'].replace([np.inf, -np.inf], np.nan)\n    return f.drop(columns='created_at')\n", encoding='utf-8')
from tyumgu_feature_rules import NUM_BASE, NUM, CAT, INPUT, make_features
F_train, F_val, F_test = [make_features(part) for part in (train,val,test)]
assert set(F_train.columns) == set(NUM + CAT)
assert all(col not in F_train.columns for col in ('late','actual_minutes','order_id'))
print('Матрица до кодирования:', F_train.shape)
print('Числовые столбцы:', NUM, '; категориальные:', CAT)

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

numeric_pipe = Pipeline([
    ('fill', SimpleImputer(strategy='median', add_indicator=True,
                           keep_empty_features=True)),
    ('scale', StandardScaler())
])
cat_pipe = Pipeline([
    ('fill', SimpleImputer(strategy='constant', fill_value='неизвестно')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocess = ColumnTransformer([
    ('num', numeric_pipe, NUM),
    ('cat', cat_pipe, CAT)
], remainder='drop', verbose_feature_names_out=True)

def build_pipeline(classifier):
    return Pipeline([
        ('derive', FunctionTransformer(make_features, validate=False)),
        ('prep', clone(preprocess)),
        ('clf', classifier)
    ])

Xtrain = train[INPUT].copy(); Xval = val[INPUT].copy(); Xtest = test[INPUT].copy()
ytrain = train['late'].astype(int).copy()
yval = val['late'].astype(int).copy()
ytest = test['late'].astype(int).copy()
assert ytrain.nunique() == 2, 'Для обучения необходимы 0 и 1 в train.'
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
print('Обучение:', Xtrain.shape, '| validation:', Xval.shape,
      '| отложенный test:', Xtest.shape)
print('Доли класса late=1 (train / validation):',
      [round(v.mean(), 3) for v in (ytrain,yval)])


## Шаг 2. Обучение только на train и вероятности на validation (15–30 мин)
Для сопоставимости фиксируем `C=1`, тот же набор признаков и `random_state=42`.

In [ ]:
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,recall_score,
                             f1_score, balanced_accuracy_score, roc_auc_score,
                             average_precision_score,log_loss,brier_score_loss,
                             PrecisionRecallDisplay, RocCurveDisplay)
import matplotlib.pyplot as plt

model=build_pipeline(LogisticRegression(C=1.0,max_iter=3000,random_state=42))
model.fit(Xtrain,ytrain)
assert list(model.named_steps['clf'].classes_)==[0,1]
val_prob=model.predict_proba(Xval)[:,1]
base=build_pipeline(DummyClassifier(strategy='prior'))
base.fit(Xtrain,ytrain)
base_prob=base.predict_proba(Xval)[:,1]
assert ((val_prob>=0)&(val_prob<=1)).all()
print('validation объектов:',len(val_prob),'положительный класс:',int(yval.sum()))


## Шаг 3. Матрица ошибок и стандартные метрики (30–45 мин)
Порядок классов фиксирован: `labels=[0,1]`; строки — факт, столбцы — прогноз. Положительный класс — задержка.

In [ ]:
def score_at_threshold(y, probs, threshold, name='LogReg'):
    pred=(np.asarray(probs)>=threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'модель':name,'порог':float(threshold),
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'Accuracy':accuracy_score(y,pred),
            'Precision':precision_score(y,pred,zero_division=0),
            'Recall':recall_score(y,pred,zero_division=0),
            'F1':f1_score(y,pred,zero_division=0),
            'BalancedAccuracy':balanced_accuracy_score(y,pred),
            'Стоимость_руб':200*fp+2000*fn}

check=pd.DataFrame([
    score_at_threshold(yval,base_prob,.5,'Dummy: prior'),
    score_at_threshold(yval,val_prob,.5,'LogReg')])
display(check.round(3))
assert (check['TN']+check['FP']+check['FN']+check['TP']==len(yval)).all()
print('Log Loss:',round(log_loss(yval,val_prob,labels=[0,1]),4))
print('Brier score:',round(brier_score_loss(yval,val_prob),4))


## Шаг 4. Три фиксированных порога и стоимость ошибок (45–60 мин)
Считайте C(t)=200·FP(t)+2000·FN(t); сравните `t=0.30`, `0.50`, `0.70`. Изменение порога не переобучает модель.

In [ ]:
fixed=pd.DataFrame([score_at_threshold(yval,val_prob,t)
                    for t in [0.30,0.50,0.70]])
display(fixed.round(3))
fixed.to_csv('10_validation_fixed_thresholds.csv',index=False)
fig, ax = plt.subplots(figsize=(7.4,4.3))
ax.plot(fixed['порог'],fixed['Стоимость_руб'],'o-')
ax.set(xlabel='Порог t',ylabel='Учебная стоимость ошибок, ₽',
       title='Стоимость ошибок для 3 заданных порогов (validation)')
ax.grid(alpha=.3); plt.tight_layout(); plt.show()


## Шаг 5. Подбор порога ТОЛЬКО на validation (60–75 мин)
Сетка порогов зафиксирована заранее: `0.05…0.95` с шагом `0.05` плюс теоретический порог для калиброванных вероятностей `200/(200+2000)≈0.0909`. При совпадении затрат выбираем больший Recall, затем порог, ближайший к 0.5. Никаких изменений после проверки test.

In [ ]:
from sklearn.metrics import confusion_matrix
C_FP, C_FN = 200, 2000
bayes_t = C_FP/(C_FP+C_FN)
thresholds=sorted(set([round(t,2) for t in np.arange(.05,1.,.05)] + [bayes_t]))
tuning=pd.DataFrame([score_at_threshold(yval,val_prob,t) for t in thresholds])
tuning['distance_to_half']=(tuning['порог']-.5).abs()
chosen=tuning.sort_values(['Стоимость_руб','Recall','distance_to_half','порог'],
                         ascending=[True,False,True,True]).iloc[0]
selected_t=float(chosen['порог'])
print('Теоретический порог при калиброванных вероятностях:',round(bayes_t,4))
print('Выбранный на validation порог:',round(selected_t,4))
display(tuning[['порог','FP','FN','Precision','Recall','F1','Стоимость_руб']].round(3))
tuning.to_csv('10_validation_threshold_grid.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4.3))
ax.plot(tuning['порог'],tuning['Стоимость_руб'],marker='.')
ax.axvline(selected_t,ls='--',label=f'Выбран t={selected_t:.3f}')
ax.axvline(bayes_t,ls=':',label=f'Теоретический t={bayes_t:.3f}')
ax.set(xlabel='Порог t',ylabel='Учебная стоимость, ₽',
       title='Выбор порога на validation')
ax.grid(alpha=.25);ax.legend();plt.tight_layout();plt.show()


## Шаг 6. ROC/PR: устойчивость ранжирования (75–80 мин)
AUC и AP оценивают ранжирование, а не стоимость конкретного решения. Если в validation только один класс, эти метрики не определены.

In [ ]:
if yval.nunique()==2:
    print('ROC AUC =',round(roc_auc_score(yval,val_prob),4))
    print('Average Precision =',round(average_precision_score(yval,val_prob),4))
    fig,axes=plt.subplots(1,2,figsize=(10,3.7))
    RocCurveDisplay.from_predictions(yval,val_prob,ax=axes[0]);axes[0].set_title('ROC / validation')
    PrecisionRecallDisplay.from_predictions(yval,val_prob,ax=axes[1]);axes[1].set_title('PR / validation')
    plt.tight_layout();plt.show()
else:
    print('ROC AUC и AP пропущены: validation содержит один класс')


## Шаг 7. Зафиксировать решение и ОДИН РАЗ проверить test (80–88 мин)
`selected_t` уже выбран; здесь **запрещено** менять список признаков, C, способ обработки данных или t на основе результатов test. Если результат разочаровал — документируйте ограничение и план нового эксперимента, но не подгоняйте итог.

In [ ]:
# Единственное обращение к итоговому тесту в блокноте.
test_prob=model.predict_proba(Xtest)[:,1]
test_rows=pd.DataFrame([
    score_at_threshold(ytest,test_prob,.5,'LogReg / фикс. 0.5'),
    score_at_threshold(ytest,test_prob,selected_t,'LogReg / выбор val')
])
display(test_rows.round(4))
test_rows.to_csv('10_final_test_once.csv',index=False)
assert (test_rows['TN']+test_rows['FP']+test_rows['FN']+test_rows['TP']==len(ytest)).all()

import joblib
joblib.dump(model,'10_final_model.joblib')
print('Зафиксированный порог =',selected_t)
print('Сохранена модель. Для её повторной загрузки понадобится tyumgu_feature_rules.py.')


## Шаг 8. Выводы и сдача (88–90 мин)
1. Для каждого из 3 порогов приведите TN/FP/FN/TP, Accuracy/Precision/Recall/F1 и стоимость.  
2. Объясните, почему минимальная цена может достигаться при невысокой Accuracy.  
3. Сравните найденный порог с 200/2200; почему они могут отличаться?  
4. Приведите ограничения: синтетические данные, размер выборок, калибровка вероятностей, гипотетические потери.  
5. В отчёте явно зафиксируйте, что test использован однократно после выбора порога.

**Сдать:** `10_student.ipynb`, отчёт №10 (DOCX/PDF), `10_validation_threshold_grid.csv`, `10_final_test_once.csv`.